In [ ]:
!pip install arcade

In [1]:
import arcade
import random
from typing import List, Tuple, Optional

In [ ]:
SCREEN_WIDTH = 1400
SCREEN_HEIGHT = 830
SCREEN_TITLE = "6nimmt! Arcade Edition by Vedanth & Sathish"

CARD_SCALE = 0.75
CARD_WIDTH = 140 * CARD_SCALE
CARD_HEIGHT = 190 * CARD_SCALE

# --- Layout Constants ---
HAND_Y_POSITION = 120
HAND_X_START = 150
HAND_X_SPACING = 110

ROW_Y_START = 350
ROW_Y_SPACING = 120
ROW_X_START = 100
ROW_X_SPACING = 110 

Card class to create card Objects:
Card Deck: 104 cards, numbered 1-104.
Bull Points: Each card has a value of "bull points" indicated on it.
Cards ending in 5: 2 bull points.
Cards ending in 0: 3 bull points.
Cards with repeating digits (11, 22, 33...): 5 bull points.
Card 55: 7 bull points (as it ends in 5 and is a repeating digit).
All other cards: 1 bull point.

In [3]:
class Card:
    def __init__(self, number: int):
        if not 1 <= number <= 104:
            raise ValueError("Card number must be between 1 and 104.")
        
        self.number = number
        self.bull_points = self._calculate_bull_points()

    def _calculate_bull_points(self) -> int:
        
        num_str = str(self.number)
        
        # Rule: Card 55 is worth 7 points
        if self.number == 55:
            return 7
        # Rule: Cards with repeating digits (11, 22...) are worth 5 points
        if len(num_str) > 1 and len(set(num_str)) == 1:
            return 5
        # Rule: Cards ending in 0 are worth 3 points
        if self.number % 10 == 0:
            return 3
        # Rule: Cards ending in 5 are worth 2 points
        if self.number % 5 == 0:
            return 2
        # Rule: All other cards are worth 1 point
        return 1

    def __repr__(self) -> str:
        """ String representation for debugging. """
        return f"Card({self.number}, BP:{self.bull_points})"


Visual Representation of card on screen

In [4]:
class CardSprite(arcade.Sprite):
    """ Represents the visual of a card on screen. """
    def __init__(self, card: Card, scale: float = 1):
        self.card = card
        
        super().__init__(scale=scale)
        self.texture = arcade.make_soft_square_texture(
            int(CARD_WIDTH / scale),
            arcade.color.WHITE,
            outer_alpha=255
        )

    def draw_card_info(self):
        """ Draws the number and bull points on the card sprite. """
        arcade.draw_text(
            str(self.card.number), self.center_x, self.center_y,
            arcade.color.BLACK, font_size=24, font_name="Arial",
            anchor_x="center", anchor_y="center", bold=True
        )
        arcade.draw_text(
            f"{self.card.bull_points} BP", self.center_x, self.center_y - CARD_HEIGHT / 3,
            arcade.color.DARK_RED, font_size=14, anchor_x="center"
        )

Player class to create Player objects
Each player object can either be a AI or Human 

In [5]:
class Player:
    """ Represents a player (human or AI). """
    def __init__(self, name: str, is_ai: bool = False):
        self.name = name
        self.hand = []
        self.score = 0
        self.is_ai = is_ai
        self.penalty_pile = []

    def add_card_to_hand(self, card: Card):
        """ Adds a single card to the player's hand. """
        self.hand.append(card)

    def calculate_score(self) -> int:
        """ Calculates the total score from the penalty pile. """
        self.score = sum(card.bull_points for card in self.penalty_pile)
        return self.score

    def __repr__(self):
        return f"Player({self.name}, Score: {self.score})"

Main game window and logic
The deck is shuffled. Each player is dealt a hand of 10 cards. Four cards are drawn from the deck and placed face-up on the table, starting four distinct rows.



In [6]:
class SixNimmtGame(arcade.Window):
    """ Main game window and logic handler. """
    def __init__(self):
        super().__init__(SCREEN_WIDTH, SCREEN_HEIGHT, SCREEN_TITLE)
        arcade.set_background_color(arcade.color.AMAZON)

        # Game data
        self.deck: List[Card] = []
        self.players: List[Player] = []
        self.rows: List[List[Card]] = [[] for _ in range(4)]
        self.turn_placements: List[Tuple[Player, Card]] = []
        
        # Sprites
        self.player_hand_sprites = arcade.SpriteList()
        self.row_sprites: List[arcade.SpriteList] = [arcade.SpriteList() for _ in range(4)]
        self.played_card_sprites = arcade.SpriteList()

        # Game state and interaction
        self.game_state = "SETUP"
        self.selected_card_sprite: Optional[CardSprite] = None
        self.placement_index = 0
        self.winner: Optional[Player] = None

    def setup(self):
        """ Sets up the game for a new round. """
        self.deck = [Card(n) for n in range(1, 105)]
        random.shuffle(self.deck)
        self.players = [Player("You")] + [Player(f"AI {i}", is_ai=True) for i in range(1, 4)]
        self.rows = [[] for _ in range(4)]
        
        self.player_hand_sprites.clear()
        for row_list in self.row_sprites: row_list.clear()
        self.played_card_sprites.clear()

        for _ in range(10):
            for player in self.players:
                player.add_card_to_hand(self.deck.pop())
        
        self.players[0].hand.sort(key=lambda c: c.number)
        for card in self.players[0].hand:
            sprite = CardSprite(card, CARD_SCALE)
            self.player_hand_sprites.append(sprite)
        self.position_player_hand()

        for i in range(4):
            card = self.deck.pop()
            self.rows[i].append(card)
            self.position_row_sprites()
        self.winner = None
        self.game_state = "PLAYER_TURN"

    def position_player_hand(self):
        """ Arranges the player's card sprites neatly on the screen. """
        for i, card_sprite in enumerate(self.player_hand_sprites):
            card_sprite.position = (HAND_X_START + i * HAND_X_SPACING, HAND_Y_POSITION)

    def position_row_sprites(self):
        """ Updates the positions of all sprites in the rows based on data. """
        for i, row_data in enumerate(self.rows):
            self.row_sprites[i].clear()
            for j, card in enumerate(row_data):
                sprite = CardSprite(card, CARD_SCALE)
                sprite.position = (
                    ROW_X_START + j * ROW_X_SPACING,
                    ROW_Y_START + i * ROW_Y_SPACING
                )
                self.row_sprites[i].append(sprite)

    def on_draw(self):
        """ Render the screen. """
        self.clear()
        arcade.draw_text("Game Rows", ROW_X_START, SCREEN_HEIGHT - 50, arcade.color.WHITE, font_size=20)
        arcade.draw_text("Your Hand", HAND_X_START, HAND_Y_POSITION + CARD_HEIGHT/2 + 20, arcade.color.WHITE, font_size=20)

        for row_list in self.row_sprites: row_list.draw()
        self.player_hand_sprites.draw()
        self.played_card_sprites.draw()

        for sprite_list in [self.player_hand_sprites, self.played_card_sprites] + self.row_sprites:
            for sprite in sprite_list:
                sprite.draw_card_info()
        self.draw_scores()

        if self.game_state == "GAME_OVER":
            self.draw_game_over()
    def draw_scores(self):
        """ Draws the scores for all players on the top right. """
        arcade.draw_text("Scores", SCREEN_WIDTH - 150, SCREEN_HEIGHT - 50, arcade.color.WHITE, font_size=20, anchor_x="center")
        for i, player in enumerate(self.players):
            score = sum(card.bull_points for card in player.penalty_pile)
            arcade.draw_text(
                f"{player.name}: {score}",
                SCREEN_WIDTH - 150,
                SCREEN_HEIGHT - 80 - i * 30,
                arcade.color.WHITE,
                font_size=16,
                anchor_x="center"
            )
    def draw_game_over(self):
        # CORRECTED LINE: Using the more compatible drawing function
        points = [
            (0, 0),  # Bottom-left
            (SCREEN_WIDTH, 0),  # Bottom-right
            (SCREEN_WIDTH, SCREEN_HEIGHT),  # Top-right
            (0, SCREEN_HEIGHT)  # Top-left
        ]
        arcade.draw_polygon_filled(points, (0, 0, 0, 150))
        
        if self.winner:
            winnerText = f"Winner: {self.winner.name} with {self.winner.score} points!"
        else:
            winnerText = "It's a tie!"

        arcade.draw_text("Game Over", SCREEN_WIDTH / 2, SCREEN_HEIGHT / 2 + 50, arcade.color.WHITE, font_size=50, anchor_x="center")
        arcade.draw_text(winnerText, SCREEN_WIDTH / 2, SCREEN_HEIGHT / 2, arcade.color.WHITE, font_size=30, anchor_x="center")
        arcade.draw_text("Click anywhere to play again.", SCREEN_WIDTH / 2, SCREEN_HEIGHT / 2 - 50, arcade.color.WHITE, font_size=20, anchor_x="center")
                
    def on_mouse_press(self, x: int, y: int, button: int, modifiers: int):
        if self.game_state == "GAME_OVER":
            self.setup()
            return

        if self.game_state != "PLAYER_TURN": return

        cards_hit = arcade.get_sprites_at_point((x, y), self.player_hand_sprites)
        if cards_hit:
            if self.selected_card_sprite:
                self.selected_card_sprite.center_y = HAND_Y_POSITION
            self.selected_card_sprite = cards_hit[0]
            self.selected_card_sprite.center_y = HAND_Y_POSITION + 20
            
            # Immediately start the turn after selection
            self.execute_turn()

    def execute_turn(self):
        """ Kicks off the logic for playing a full turn. """
        if not self.selected_card_sprite: return
        self.game_state = "PROCESSING_TURN"

        # 1. Gather all played cards
        player_card = self.selected_card_sprite.card
        self.turn_placements = [(self.players[0], player_card)]
        self.players[0].hand.remove(player_card)
        self.selected_card_sprite.kill() # Remove sprite from hand

        for player in self.players:
            if player.is_ai:
                ai_card = self._choose_card_for_ai(player)
                self.turn_placements.append((player, ai_card))
                player.hand.remove(ai_card)

        # 2. Sort cards by number
        self.turn_placements.sort(key=lambda p: p[1].number)

        # 3. Start placing them sequentially
        self.placement_index = 0
        arcade.schedule(self._process_next_placement, 1.0) # Place one card every second

    def _process_next_placement(self, delta_time: float):
        """ Places one card on the board from the sorted turn_placements list. """
        if self.placement_index >= len(self.turn_placements):
            arcade.unschedule(self._process_next_placement)
            if not self.players[0].hand: # Check if round is over
                self.game_state = "GAME_OVER"
                for p in self.players:
                    p.calculate_score()
                
                min_score = min(p.score for p in self.players)
                winners = [p for p in self.players if p.score == min_score]

                if len(winners) == 1:
                    self.winner = winners[0]
                else: 
                    self.winner = None 
                
                print(f"Game over! Minimum score was {min_score}.")

            else:
                self.game_state = "PLAYER_TURN"
            return

        player, card = self.turn_placements[self.placement_index]
        
        # --- Find target row ---
        target_row_idx = -1
        valid_rows = []
        for i, row in enumerate(self.rows):
            if row[-1].number < card.number:
                valid_rows.append((i, row[-1].number))

        if valid_rows:
            target_row_idx = max(valid_rows, key=lambda item: item[1])[0]
        
        # --- Place the card ---
        if target_row_idx != -1: 
            row = self.rows[target_row_idx]
            if len(row) == 5: 
                print(f"{player.name} takes row {target_row_idx + 1} with {card}")
                player.penalty_pile.extend(row)
                self.rows[target_row_idx] = [card]
            else: 
                row.append(card)
        else: 
            bull_points_per_row = [sum(c.bull_points for c in r) for r in self.rows]
            chosen_row_idx = bull_points_per_row.index(min(bull_points_per_row))
            print(f"{player.name}'s card {card} is too low, takes row {chosen_row_idx + 1}")
            
            player.penalty_pile.extend(self.rows[chosen_row_idx])
            self.rows[chosen_row_idx] = [card]
            
        self.position_row_sprites() # Update visuals
        self.placement_index += 1

    def _choose_card_for_ai(self, player: Player) -> Card:
        """ AI heuristic logic to select a card. """
        options = []
        for card in player.hand:
            valid_rows = []
            for i, row in enumerate(self.rows):
                if row[-1].number < card.number:
                    valid_rows.append((i, row))
            
            if not valid_rows:
                bull_points_per_row = [sum(c.bull_points for c in r) for r in self.rows]
                penalty = min(bull_points_per_row)
                options.append({'card': card, 'penalty': penalty, 'is_safe': False})
            else:
                target_row_tuple = max(valid_rows, key=lambda item: item[1][-1].number)
                target_row = self.rows[target_row_tuple[0]]
                if len(target_row) < 5:
                    options.append({'card': card, 'penalty': 0, 'is_safe': True})
                else:
                    penalty = sum(c.bull_points for c in target_row)
                    options.append({'card': card, 'penalty': penalty, 'is_safe': False})
        
        safe_options = [opt for opt in options if opt['is_safe']]
        if safe_options:
            return max(safe_options, key=lambda opt: opt['card'].number)['card']
        else:
            return min(options, key=lambda opt: opt['penalty'])['card']

Main function

In [7]:
def main():
    """ Main method """
    window = SixNimmtGame()
    window.setup()
    arcade.run()


if __name__ == "__main__":
    main()

c:\Users\raman\OneDrive\Documents\FAI\game\.venv\Lib\site-packages\arcade\exceptions.py:138: PerformanceWarning: draw_text is an extremely slow function for displaying text. Consider using Text objects instead.
  warnings.warn(message, warning_type)


AI 2's card Card(19, BP:1) is too low, takes row 2
AI 3's card Card(9, BP:1) is too low, takes row 3
AI 2's card Card(10, BP:3) is too low, takes row 3
You's card Card(17, BP:1) is too low, takes row 3
You's card Card(13, BP:1) is too low, takes row 3
AI 3's card Card(6, BP:1) is too low, takes row 1
Game over! Minimum score was 0.
